# 47. 물리 기반 합성 불량 주입 → 탐지 한계(최소 탐지 크기) 측정

**아이디어**: 실제 불량 에피소드가 1개뿐이면 실제 라벨로는 '어떤 이상을 잡을 수 있나'를 말할 수 없음 →
정상 샷에 **사출 물리로 정한 불량 패턴**을 크기별로 넣고 몇 %를 잡는지 측정 (22번의 일반 이상치 주입과 달리 메커니즘별)

| 패턴 | 바뀌는 변수 | 모양 |
|---|---|---|
| 미성형_재료부족 | 충전시간↓ 사출속도↑ 사출압↓ 쿠션↓ | 계단 |
| 미성형_냉각점도 | 금형온도3↓ 사출압↑ 충전시간↑ 사출속도↓ | 램프(서서히) |
| 가스_과열 | 배럴2~5↑ 호퍼↑ 사이클↑ | 램프 |
| 가스_공급불안정 | 배럴2·호퍼·쿠션·배럴5·계량시간 흔들림 | 잡음 |
| 가스_계량지연 | 계량시간↑ 사이클↑ | 계단 |

- 크기 = 변수별 **같은 Run 안 정상 변동**(Run 안 IQR/1.349)의 a배 (0.5~8)
- 정상 기준 데이터에서 150개 위치를 골라 20샷 주입 (모든 패턴·크기에 같은 위치) → 최종 모델과 같은 파생변수·점수
- `real_defect_magnitude`: **실제 불량 샷이 같은 단위로 얼마나 벗어났는지** → 곡선에 실제 불량 위치를 대 볼 수 있음
- 실행 시간: 제품당 약 2~5분

In [ ]:
import pandas as pd
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40); pd.set_option('display.max_rows', 200)
%load_ext autoreload
%autoreload 2
import synthetic_injection as SI
R = SI.run(products=('cn7', 'rg3'), n_ep=150)

## 최소 탐지 크기 — 에피소드 절반 이상을 잡는 첫 크기 (정상 변동의 몇 배)

In [ ]:
display(R['mde'])

## 패턴·크기별 탐지율 (점선 = 주입 안 한 같은 구간의 오경보)

In [ ]:
SI.plot(R, 'cn7')
SI.plot(R, 'rg3')

In [ ]:
T = R['table']
show = ['제품', '패턴', '크기(a×정상변동)', '탐지기', '에피소드 탐지율', '샷 탐지율', '첫 탐지까지 샷(중앙값)', '앞 5샷 탐지율', '뒤 5샷 탐지율']
display(T[T['탐지기'].isin(['최종 모델 q0.999', 'std4 q0.999', 'CUSUM(최종) 경보율1%'])][show].round(3))

## 실제 불량은 정상 변동의 몇 배였나 (부호 포함 중앙값 = 패턴 방향이 맞는지도 확인)

In [ ]:
RD = R['real']
display(RD.pivot_table(index=['제품', '변수'], columns='유형', values='편차 중앙값(정상변동 배수)').round(2))
display(RD.pivot_table(index=['제품', '변수'], columns='유형', values='|편차| 중앙값').round(2))

## 읽는 법
1. **최소 탐지 크기**: 예) 미성형_재료부족이 최종 모델로 2배부터 잡힌다면 '정상 변동의 2배 이상 재료 부족은 탐지 가능' → 보고서의 '탐지 범위' 근거
2. **앞 5샷 vs 뒤 5샷**: 계단형은 앞에서 잡히고 뒤에서 떨어지면 → 인과적 기준선(직전 20샷)이 새 수준에 적응한 것 = **지속 이상은 순차 감시(46번)·열 상태 변수가 필요**한 근거
3. **실제 불량 크기표**: 실제 가스 샷의 편차가 1배 안팎이면 → 최소 탐지 크기보다 작아 **원리적으로 단발 탐지가 어렵다**는 설명 근거 (모델 한계의 정량화)
4. 부호가 패턴과 반대면 그 메커니즘은 이 데이터의 불량 원인이 아닐 가능성